# 07 - K-nearest-source ablation

Public research notebook for the STARS workflow. Dataset files and generated artifacts are intentionally excluded from Git. See `../data/README.md` before running.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path("..").resolve()
DATA_DIR = REPO_ROOT / "data"
RESULTS_DIR = REPO_ROOT / "results"
(DATA_DIR / "raw").mkdir(parents=True, exist_ok=True)
(DATA_DIR / "interim").mkdir(parents=True, exist_ok=True)
(DATA_DIR / "processed").mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO_ROOT / "src"))


In [ ]:
# build_windows_per_building.py
# ----------------------------------------------------------------------
# • One SplitDataset (and DataLoader) for every distinct building_id
# • Same sliding-window logic as before
# ----------------------------------------------------------------------

import pandas as pd, numpy as np, torch
from datetime import datetime, timedelta
from torch.utils.data import Dataset, DataLoader

# ─────────────────────────────────────────────────────────────
# 1.  Sliding-window dataset (unchanged)
# ─────────────────────────────────────────────────────────────
FEATURE_COLS = [
    "electricity_kWh",
    "airTemperature", "relativeHumidity", "windSpeed",
    "month_of_year", "day_of_month", "day_of_week", "hour_of_day",
]

class SplitDataset(Dataset):
    def __init__(self, df, past_len=168, future_len=24, stride=1):
        self.x, self.y = self._build_windows(df, past_len, future_len, stride)

    # --- helpers for date math ---
    @staticmethod
    def _first_monday_july(year: int) -> datetime:
        d = datetime(year, 7, 1)
        while d.weekday() != 0:  # 0 = Monday
            d += timedelta(days=1)
        return d.replace(hour=0, minute=0, second=0, microsecond=0)

    @staticmethod
    def _august_end(year: int) -> datetime:
        return datetime(year, 8, 31, 23, 59, 59)

    # --- core window builder ---
    def _build_windows(self, df, past_len, future_len, stride):
        xs, ys = [], []

        # each df is already a **single** building
        grp = df.sort_values("timestamp").reset_index(drop=True)

        # fill NaNs by nearest neighbour interpolation
        grp_feat = (
            grp[FEATURE_COLS]
            .interpolate(method="nearest", limit_direction="both")
        )

        times = grp["timestamp"].to_numpy()
        data  = grp_feat.to_numpy(np.float32)
        if len(times) == 0:
            return self._empty(past_len, future_len)

        year   = pd.Timestamp(times[0]).year
        anchor = self._first_monday_july(year)
        cutoff = self._august_end(year)

        i = np.searchsorted(times, np.datetime64(anchor), side="left")

        while (
            i + past_len + future_len <= len(data)
            and pd.Timestamp(times[i]).to_pydatetime() <= cutoff
        ):
            past   = data[i : i + past_len]
            future = data[i + past_len : i + past_len + future_len]
            xs.append(past)
            ys.append(future)
            i += stride

        if xs:
            return np.stack(xs), np.stack(ys)
        return self._empty(past_len, future_len)

    @staticmethod
    def _empty(past_len, future_len):
        return (
            np.empty((0, past_len, len(FEATURE_COLS)), np.float32),
            np.empty((0, future_len, len(FEATURE_COLS)), np.float32),
        )

    # torch boiler-plate
    def __len__(self):          return len(self.x)
    def __getitem__(self, idx): return (
        torch.from_numpy(self.x[idx]),
        torch.from_numpy(self.y[idx]),
    )

# ─────────────────────────────────────────────────────────────
# 2.  One DataLoader per building_id
# ─────────────────────────────────────────────────────────────
def make_building_dataloaders(
    data_csv,
    past_len    = 168,
    future_len  = 24,
    stride      = 1,
    batch_size  = 64,
    num_workers = 0,
):
    df = (
        pd.read_csv(data_csv, parse_dates=["timestamp"])
          .sort_values(["building_id", "timestamp"])
    )

    loaders = {}
    for bid, grp in df.groupby("building_id", sort=False):
        ds = SplitDataset(grp, past_len, future_len, stride)
        if len(ds) == 0:
            # optional: print or log empty buildings
            continue

        loaders[bid] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers,
            pin_memory=True,
        )

    return loaders

# ─────────────────────────────────────────────
# 3.  Example usage
# ─────────────────────────────────────────────
test_loaders = make_building_dataloaders(
    data_csv   = str(DATA_DIR / "processed" / "bdg2_ny_2017_with_weather.csv"),
    past_len   = 336,
    future_len = 24,
    stride     = 24,
    batch_size = 64,
)

# Peek at one building
sample_bid = next(iter(test_loaders))
x, y = next(iter(test_loaders[sample_bid]))
print(f"Building {sample_bid} ➜ x:", x.shape, " y:", y.shape)
# e.g. (64, 336, 8)  (64, 24, 8)


In [ ]:
train_loaders = make_building_dataloaders(
    data_csv   = str(DATA_DIR / "processed" / "combined_ready_raw.csv"),
    past_len   = 336,
    future_len = 24,
    stride     = 12,
    batch_size = 64,
)

# Peek at one building
sample_bid = next(iter(train_loaders))
x, y = next(iter(train_loaders[sample_bid]))
print(f"Building {sample_bid} ➜ x:", x.shape, " y:", y.shape)
# e.g. (64, 336, 8)  (64, 24, 8)

In [ ]:
# build_windows_per_building_first_only.py
# ----------------------------------------------------------------------
# • ONE SplitDataset + DataLoader PER building_id
# • Keeps **only the first** window that begins at the first Monday
#   (00:00) of July and spans [past_len] hours of history +
#   [future_len] hours of forecast.
# ----------------------------------------------------------------------

import pandas as pd
import numpy as np
import torch
from datetime import datetime, timedelta
from torch.utils.data import Dataset, DataLoader

# ----------------------------------------------------------------------
# 1.  Sliding-window dataset  (one sample per building)
# ----------------------------------------------------------------------
FEATURE_COLS = [
    "electricity_kWh",
    "airTemperature", "relativeHumidity", "windSpeed",
    "month_of_year", "day_of_month", "day_of_week", "hour_of_day",
]

class SplitDataset(Dataset):
    """
    Holds exactly ONE (x, y) pair for a single building:
        x : [past_len,  n_features]
        y : [future_len, n_features]
    """
    def __init__(self, df, past_len=168, future_len=24):
        self.x, self.y = self._first_window(df, past_len, future_len)

    # ─ helpers for anchoring dates ──────────────────────────
    @staticmethod
    def _first_monday_july(year: int) -> datetime:
        d = datetime(year, 7, 1)
        while d.weekday() != 0:                  # 0 = Monday
            d += timedelta(days=1)
        return d.replace(hour=0, minute=0, second=0, microsecond=0)

    @staticmethod
    def _august_end(year: int) -> datetime:
        return datetime(year, 8, 31, 23, 59, 59)

    # ─ core: pick the first valid window ───────────────────
    def _first_window(self, df, past_len, future_len):
        # sort & fill gaps
        grp = df.sort_values("timestamp").reset_index(drop=True)

        grp_feat = (
            grp[FEATURE_COLS]
            .interpolate(method="nearest", limit_direction="both")
        )

        times = grp["timestamp"].to_numpy()          # numpy datetime64
        data  = grp_feat.to_numpy(np.float32)

        if len(times) == 0:
            return self._empty(past_len, future_len)

        year   = pd.Timestamp(times[0]).year
        anchor = self._first_monday_july(year)
        cutoff = self._august_end(year)

        i = np.searchsorted(times, np.datetime64(anchor), side="left")

        if (
            i + past_len + future_len <= len(data) and
            pd.Timestamp(times[i]).to_pydatetime() <= cutoff
        ):
            past   = data[i : i + past_len]
            future = data[i + past_len : i + past_len + future_len]
            return past[None, ...], future[None, ...]   # keep batch dim

        return self._empty(past_len, future_len)

    @staticmethod
    def _empty(past_len, future_len):
        return (
            np.empty((0, past_len, len(FEATURE_COLS)), np.float32),
            np.empty((0, future_len, len(FEATURE_COLS)), np.float32),
        )

    # torch boiler-plate
    def __len__(self):          return len(self.x)      # 0 or 1
    def __getitem__(self, idx): return (
        torch.from_numpy(self.x[idx]),
        torch.from_numpy(self.y[idx]),
    )

# ----------------------------------------------------------------------
# 2.  One DataLoader per building_id
# ----------------------------------------------------------------------
def make_building_dataloaders(
    data_csv,
    past_len    = 168,
    future_len  = 24,
    batch_size  = 1,      # one sample ⇒ batch_size = 1 is natural
    num_workers = 0,
):
    df = (
        pd.read_csv(data_csv, parse_dates=["timestamp"])
          .sort_values(["building_id", "timestamp"])
    )

    loaders = {}
    for bid, grp in df.groupby("building_id", sort=False):
        ds = SplitDataset(grp, past_len, future_len)
        if len(ds) == 0:            # no valid July-Aug window
            continue

        loaders[bid] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=False,          # only one item
            num_workers=num_workers,
            pin_memory=True,
        )

    return loaders

# ----------------------------------------------------------------------
# 3.  Example usage
# ----------------------------------------------------------------------

loaders = make_building_dataloaders(
    data_csv   = str(DATA_DIR / "processed" / "bdg2_ny_2017_with_weather.csv"),
    past_len   = 336,
    future_len = 24,
)

sample_bid = next(iter(loaders))
xb, yb = next(iter(loaders[sample_bid]))
print(f"Building {sample_bid} ➜ x:", xb.shape, " y:", yb.shape)
# → (1, 336, 8)  (1, 24, 8)


In [ ]:
from models.PatchTST import Model          # import the class you pasted

# 1) Configuration object -------------------------------------------------
class CFG:
    # core task & data lengths
    task_name = "long_term_forecast"   # or short_term_forecast / imputation …
    seq_len   = 336                    # look-back steps L
    pred_len  = 24                     # forecast horizon  T
    enc_in    = 8                      # total historical channels
    c_out     = 1                      # predict only the 1st channel
    features  = "MS"                   # multi-in, single-out

    # transformer backbone
    d_model   = 32
    d_ff      = 64
    n_heads   = 4
    e_layers  = 3
    dropout   = 0.1
    activation= "gelu"

    # other args required by EncoderLayer signatures
    factor     = 3
    top_k      = 3
    num_kernels= 6

cfg = CFG()
# ── device and model ──────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# ── build one PatchTST per cluster ──────────────────────────────────
model = Model(cfg, patch_len=16, stride=8).to(DEVICE) 

print(model)
print("\nPatchTST Model created. #Params:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
import json

knn_dict = json.load(open(str(DATA_DIR / "interim" / "knn_dict_raw.json")  ))
knn_dist = json.load(open(str(DATA_DIR / "interim" / "knn_dist_raw.json")  ))

In [ ]:
import itertools

model = Model(cfg, patch_len=16, stride=8).to(DEVICE) 

test_bid  = next(iter(knn_dict))          # first key
train_bids= knn_dict[test_bid][:5]        # five nearest training IDs

print("Test building :", test_bid)
print("Train set IDs :", train_bids)

In [ ]:
from torch.utils.data import ConcatDataset, DataLoader
# ─────────────────────────────────────────────────────────────
# 1)  TEST loader   (already built)
# ─────────────────────────────────────────────────────────────
test_dl = test_loaders[test_bid]

# ─────────────────────────────────────────────────────────────
# 2)  TRAIN loader = concat of the 5 neighbours
# ─────────────────────────────────────────────────────────────
# Filter out any IDs that might be missing from train_loaders
train_ds_list = [
    train_loaders[bid].dataset
    for bid in train_bids
    if bid in train_loaders
]

if not train_ds_list:
    raise ValueError("None of the nearest-neighbour IDs are in train_loaders.")

concat_ds = ConcatDataset(train_ds_list)

# reuse batch_size / num_workers from one of the individual loaders
tmpl_loader = train_loaders[train_bids[0]]
train_dl = DataLoader(
    concat_ds,
    batch_size   = tmpl_loader.batch_size,
    shuffle      = True,
    num_workers  = tmpl_loader.num_workers,
    pin_memory   = tmpl_loader.pin_memory,
)

print(f"train_dl batches : {len(train_dl)}")
print(f"test_dl  batches : {len(test_dl)}")

In [ ]:
from tqdm import tqdm
import math, torch, torch.nn as nn
# ─────────────────────────────────────────────
# Helpers for CVRMSE • NMBE • MAE (p = 1)
# ─────────────────────────────────────────────
def _accum(out, y):
    err   = out - y
    sse   = torch.sum(err ** 2).item()        # Σ(err²)
    bias  = torch.sum(err).item()             # Σ(err)
    sum_y = torch.sum(y).item()               # Σ(y)
    sae   = torch.sum(torch.abs(err)).item()  # Σ|err|
    n     = y.numel()
    return sse, bias, sum_y, sae, n

def _final(sse, bias, sum_y, sae, n):
    denom  = n - 1
    rmse   = math.sqrt(sse / denom)
    mean_y = sum_y / n
    cvrmse = 100.0 * rmse / mean_y
    nmbe   = 100.0 * bias / (denom * mean_y)
    mae    = sae / n
    return cvrmse, nmbe, mae

EPOCHS = 100
LR     = 1e-3

optim   = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.MSELoss()

for epoch in range(1, EPOCHS + 1):                 # <- lowercase loop var
    # ---------- TRAIN ----------
    model.train()
    tr_sse = tr_bias = tr_sumy = tr_sae = tr_n = 0

    for xb, yb in tqdm(train_dl, leave=False, desc=f"[train] e{epoch:02d}"):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        pred   = model(xb, None, None, None)[..., 0]
        loss   = loss_fn(pred, yb[..., 0])

        optim.zero_grad()
        loss.backward()
        optim.step()

        se, bias, sumy, sae, n = _accum(pred, yb[..., 0])
        tr_sse += se; tr_bias += bias; tr_sumy += sumy; tr_sae += sae; tr_n += n

    tr_cvrmse, tr_nmbe, tr_mae = _final(tr_sse, tr_bias, tr_sumy, tr_sae, tr_n)

    # ---------- VALID ----------
    model.eval()
    ts_sse = ts_bias = ts_sumy = ts_sae = ts_n = 0
    with torch.no_grad():
        for xb, yb in tqdm(test_dl, leave=False, desc=f"[valid] e{epoch:02d}"):
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred   = model(xb, None, None, None)[..., 0]
            se, bias, sumy, sae, n = _accum(pred, yb[..., 0])
            ts_sse += se; ts_bias += bias; ts_sumy += sumy; ts_sae += sae; ts_n += n

    ts_cvrmse, ts_nmbe, ts_mae = _final(ts_sse, ts_bias, ts_sumy, ts_sae, ts_n)

    # ---------- LOG ----------
    print(
        f"Epoch {epoch:02d} | "
        f"Train CVRMSE {tr_cvrmse:6.2f}%  NMBE {tr_nmbe:6.2f}%  MAE {tr_mae:8.4f} | "
        f"Valid CVRMSE {ts_cvrmse:6.2f}%  NMBE {ts_nmbe:6.2f}%  MAE {ts_mae:8.4f}"
    )

In [ ]:
from torch.utils.data import ConcatDataset, DataLoader
from tqdm import tqdm
import math, torch, torch.nn as nn

for K_NEIGHBOUR in [40, 80, 160]:
    print(K_NEIGHBOUR)
    # ------------------------------------------------------------------
    # 0)  Hyper-parameters
    # ------------------------------------------------------------------
    EPOCHS      = 30
    LR          = 1e-3
    DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # ------------------------------------------------------------------
    # 1)  Build loaders for a single test building
    # ------------------------------------------------------------------
    def build_loaders(test_bid):
        test_dl  = loaders[test_bid] #test_loaders
    
        train_ids = [bid for bid in knn_dict[test_bid][:K_NEIGHBOUR]
                     if bid in train_loaders]
        train_dists = knn_dist[test_bid][:K_NEIGHBOUR]
        if not train_ids:
            raise ValueError(f"No neighbours in train_loaders for {test_bid}")
    
        concat_ds   = ConcatDataset([train_loaders[bid].dataset for bid in train_ids])
        template_dl = train_loaders[train_ids[0]]
        train_dl    = DataLoader(
            concat_ds,
            batch_size  = template_dl.batch_size,
            shuffle     = True,
            num_workers = template_dl.num_workers,
            pin_memory  = template_dl.pin_memory,
        )
        return train_dl, test_dl, train_ids, train_dists
    
    # ------------------------------------------------------------------
    # 2)  Helpers for metrics (unchanged)
    # ------------------------------------------------------------------
    def _accum(out, y):
        err   = out - y
        sse   = torch.sum(err ** 2).item()
        bias  = torch.sum(err).item()
        sum_y = torch.sum(y).item()
        sae   = torch.sum(torch.abs(err)).item()
        n     = y.numel()
        return sse, bias, sum_y, sae, n
    
    def _final(sse, bias, sum_y, sae, n):
        rmse   = math.sqrt(sse / (n - 1))
        mean_y = sum_y / n
        cvrmse = 100.0 * rmse / mean_y
        nmbe   = 100.0 * bias / ((n - 1) * mean_y)
        mae    = sae / n
        return cvrmse, nmbe, mae
    
    # ------------------------------------------------------------------
    # 3)  One-epoch train/valid loop
    # ------------------------------------------------------------------
    def train_one_epoch(model, train_dl, test_dl, optim, loss_fn, epoch):
        # ---- train ----
        model.train()
        tr_sse = tr_bias = tr_sumy = tr_sae = tr_n = 0
        for xb, yb in tqdm(train_dl, leave=False, desc=f"[train] e{epoch:02d}"):
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred   = model(xb, None, None, None)[..., 0]
            loss   = loss_fn(pred, yb[..., 0])
    
            optim.zero_grad()
            loss.backward()
            optim.step()
    
            se, bias, sumy, sae, n = _accum(pred, yb[..., 0])
            tr_sse += se; tr_bias += bias; tr_sumy += sumy; tr_sae += sae; tr_n += n
    
        # ---- valid ----
        model.eval()
        ts_sse = ts_bias = ts_sumy = ts_sae = ts_n = 0
        with torch.no_grad():
            for xb, yb in tqdm(test_dl, leave=False, desc=f"[valid] e{epoch:02d}"):
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                pred   = model(xb, None, None, None)[..., 0]
                se, bias, sumy, sae, n = _accum(pred, yb[..., 0])
                ts_sse += se; ts_bias += bias; ts_sumy += sumy; ts_sae += sae; ts_n += n
    
        tr = _final(tr_sse, tr_bias, tr_sumy, tr_sae, tr_n)
        ts = _final(ts_sse, ts_bias, ts_sumy, ts_sae, ts_n)
        return tr, ts
    
    # ------------------------------------------------------------------
    # 4)  Iterate over ALL test buildings
    # ------------------------------------------------------------------
    results = []                      # collect per-building metrics
    
    for test_bid in knn_dict.keys():
        # 4-a. loaders
        train_dl, test_dl, neighbour_ids, neighbour_dists = build_loaders(test_bid)
    
        # 4-b. new model instance  (replace ModelClass() with yours)
        model = Model(cfg, patch_len=16, stride=8).to(DEVICE) 
        optim = torch.optim.Adam(model.parameters(), lr=LR)
        loss_fn = nn.MSELoss()
    
        print(f"\n▶ Building {test_bid}  (train from neighbours {neighbour_ids})")
        for epoch in range(1, EPOCHS + 1):
            tr_metrics, ts_metrics = train_one_epoch(
                model, train_dl, test_dl, optim, loss_fn, epoch
           )
#            print(
#                f"Epoch {epoch:02d} | "
#                f"Train CVRMSE {tr_metrics[0]:6.2f}%  NMBE {tr_metrics[1]:6.2f}%  MAE {tr_metrics[2]:8.4f} | "
#                f"Valid CVRMSE {ts_metrics[0]:6.2f}%  NMBE {ts_metrics[1]:6.2f}%  MAE {ts_metrics[2]:8.4f}"
#            )
    
        # 4-c. keep final metrics
        results.append({
            "test_building" : test_bid,
            "train_ids"     : neighbour_ids,
            "train_dists"   : neighbour_dists,
            "val_CVRMSE(%)" : ts_metrics[0],
            "val_NMBE(%)"   : ts_metrics[1],
            "val_MAE"       : ts_metrics[2],
        })
    
    # ------------------------------------------------------------------
    # 5)  results is now a list of dicts you can turn into a DataFrame
    # ------------------------------------------------------------------
    df_results = pd.DataFrame(results)
    print(df_results.head())
    
    file_path = str(RESULTS_DIR / ("results_k_1day" + str(K_NEIGHBOUR) + ".csv"))   # pick any path you like
    df_results.to_csv(file_path, index=False)
    print(f"Results saved → {file_path}")

In [ ]:
# train_all_then_test.py  ▸  train on ALL train_loaders, test on each test_loader
# ──────────────────────────────────────────────────────────────────────────────
from torch.utils.data import ConcatDataset, DataLoader
from tqdm import tqdm
import torch, torch.nn as nn, math, pandas as pd

# ─────────────────────────────────────────────────────────────
# 0)  Hyper-params
# ─────────────────────────────────────────────────────────────
EPOCHS  = 30
LR      = 1e-3
DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH   = next(iter(train_loaders.values())).batch_size
N_WORK  = next(iter(train_loaders.values())).num_workers
PINMEM  = next(iter(train_loaders.values())).pin_memory

# ─────────────────────────────────────────────────────────────
# 1)  Build *one* global training DataLoader
# ─────────────────────────────────────────────────────────────
train_concat_ds = ConcatDataset([ldr.dataset for ldr in train_loaders.values()])

train_dl = DataLoader(
    train_concat_ds,
    batch_size  = BATCH,
    shuffle     = True,
    num_workers = N_WORK,
    pin_memory  = PINMEM,
)
print(f"Global train-set: {len(train_concat_ds):,} windows  "
      f"({len(train_loaders)} buildings)")

# ─────────────────────────────────────────────────────────────
# 2)  Metric helpers (unchanged)
# ─────────────────────────────────────────────────────────────
def _accum(out, y):
    err   = out - y
    return (
        torch.sum(err ** 2).item(),          # sse
        torch.sum(err).item(),               # bias
        torch.sum(y).item(),                 # sum_y
        torch.sum(torch.abs(err)).item(),    # sae
        y.numel(),                           # n
    )

def _final(sse, bias, sum_y, sae, n):
    rmse   = math.sqrt(sse / (n - 1))
    mean_y = sum_y / n
    return (
        100.0 * rmse / mean_y,               # CVRMSE %
        100.0 * bias / ((n - 1) * mean_y),   # NMBE %
        sae / n,                             # MAE
    )

# ─────────────────────────────────────────────────────────────
# 3)  Instantiate & train once
# ─────────────────────────────────────────────────────────────
model   = Model(cfg, patch_len=16, stride=8).to(DEVICE)
optim   = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.MSELoss()

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_sse = tr_bias = tr_sumy = tr_sae = tr_n = 0
    for xb, yb in tqdm(train_dl, desc=f"[train] e{epoch:02d}", leave=False):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        pred   = model(xb, None, None, None)[..., 0]
        loss   = loss_fn(pred, yb[..., 0])

        optim.zero_grad(); loss.backward(); optim.step()

        se, b, sy, sae, n = _accum(pred, yb[..., 0])
        tr_sse += se; tr_bias += b; tr_sumy += sy; tr_sae += sae; tr_n += n

    cvrm, nmbe, mae = _final(tr_sse, tr_bias, tr_sumy, tr_sae, tr_n)
    print(f"Epoch {epoch:02d}  •  Train CVRMSE {cvrm:6.2f} %  "
          f"NMBE {nmbe:6.2f} %  MAE {mae:8.4f}")

In [ ]:
full_model = model

In [ ]:
# ─────────────────────────────────────────────────────────────
# 4)  Evaluate per test building
# ─────────────────────────────────────────────────────────────
results = []

full_model.eval()
with torch.no_grad():
    for test_bid, test_dl in loaders.items():
        ts_sse = ts_bias = ts_sumy = ts_sae = ts_n = 0
        for xb, yb in test_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred   = full_model(xb, None, None, None)[..., 0]
            se, b, sy, sae, n = _accum(pred, yb[..., 0])
            ts_sse += se; ts_bias += b; ts_sumy += sy; ts_sae += sae; ts_n += n

        cvrm, nmbe, mae = _final(ts_sse, ts_bias, ts_sumy, ts_sae, ts_n)
        results.append({
            "test_building" : test_bid,
            "val_CVRMSE(%)" : cvrm,
            "val_NMBE(%)"   : nmbe,
            "val_MAE"       : mae,
        })
        print(f"✓ {test_bid:>8}  →  CVRMSE {cvrm:6.2f}%  NMBE {nmbe:6.2f}%  MAE {mae:8.4f}")

# ─────────────────────────────────────────────────────────────
# 5)  Save results
# ─────────────────────────────────────────────────────────────
df_results = pd.DataFrame(results)
csv_path   = str(RESULTS_DIR / "results_all_train_1day.csv")
df_results.to_csv(csv_path, index=False)
print(f"\nSaved per-building metrics → {csv_path}")
